In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path().resolve().parent
csv_path = BASE_DIR / "data" / "processed" / "imdb_with_sentiment.csv"

df = pd.read_csv(csv_path)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Shape: (250, 24)
Columns: ['rank', 'title', 'year', 'rating', 'votes', 'genre', 'runtime_minutes', 'imdb_id', 'imdb_url', 'weighted_score', 'plot', 'awards', 'metascore', 'imdb_rating', 'box_office', 'director', 'language', 'country', 'sentiment_compound', 'sentiment_positive', 'sentiment_negative', 'sentiment_neutral', 'sentiment_intensity', 'sentiment_label']


,rank,title,year,rating,votes,genre,runtime_minutes,imdb_id,imdb_url,weighted_score,...,box_office,director,language,country,sentiment_compound,sentiment_positive,sentiment_negative,sentiment_neutral,sentiment_intensity,sentiment_label
0,1,The Shawshank Redemption,1994,9.3,3195314,Drama,142,tt0111161,https://www.imdb.com/title/tt0111161/,9.1798,...,"$28,767,189",Frank Darabont,English,United States,-0.8962,0.107,0.258,0.635,0.8962,negative
1,2,The Godfather,1972,9.2,2229956,"Crime, Drama",175,tt0068646,https://www.imdb.com/title/tt0068646/,9.0403,...,"$136,381,073",Francis Ford Coppola,"English, Italian, Latin",United States,0.4785,0.083,0.052,0.865,0.4785,positive
2,3,The Dark Knight,2008,9.1,3176249,"Crime, Thriller",152,tt0468569,https://www.imdb.com/title/tt0468569/,8.9910,...,"$534,987,076",Christopher Nolan,"English, Mandarin","United States, United Kingdom",-0.3612,0.146,0.160,0.695,0.3612,negative


In [2]:
# Convert votes to numeric (remove commas if any)
df["votes"] = pd.to_numeric(df["votes"].astype(str).str.replace(",", ""), errors="coerce")

# Convert year to numeric
df["year"] = pd.to_numeric(df["year"], errors="coerce")

# Convert rating to numeric
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# Convert metascore to numeric
df["metascore"] = pd.to_numeric(df["metascore"], errors="coerce")

# Drop rows with missing sentiment or votes
df = df.dropna(subset=["sentiment_compound", "votes", "year", "rating"])

print(f"Clean dataset: {len(df)} movies")
print(f"\nData types:")
print(df[["votes", "year", "rating", "sentiment_compound"]].dtypes)

Clean dataset: 250 movies

Data types:
votes                   int64
year                    int64
rating                float64
sentiment_compound    float64
dtype: object


In [3]:
# ============================================================
# LONGEVITY SCORE
# Logic: A movie released in 1994 with 2.8M votes today means
# people are STILL actively watching and rating it decades later.
# We normalize votes by movie age to get a fair comparison.
# A 2020 movie naturally has fewer votes than a 1994 movie.
# ============================================================

CURRENT_YEAR = 2025

# Age of movie in years (minimum 1 to avoid division by zero)
df["age_years"] = (CURRENT_YEAR - df["year"]).clip(lower=1)

# Votes per year of existence — core longevity metric
df["votes_per_year"] = df["votes"] / df["age_years"]

# Normalize to 0-100 scale for readability
df["longevity_score"] = (
    (df["votes_per_year"] - df["votes_per_year"].min()) /
    (df["votes_per_year"].max() - df["votes_per_year"].min())
) * 100

df["longevity_score"] = df["longevity_score"].round(2)

print("Longevity Score Stats:")
print(df["longevity_score"].describe().round(2))
print(f"\nHighest longevity: {df.loc[df['longevity_score'].idxmax(), 'title']}")
print(f"Lowest longevity:  {df.loc[df['longevity_score'].idxmin(), 'title']}")

Longevity Score Stats:
count    250.00
mean       7.61
std       10.35
min        0.00
25%        1.85
50%        4.89
75%        9.58
max      100.00
Name: longevity_score, dtype: float64

Highest longevity: Dune: Part Two
Lowest longevity:  Metropolis


In [4]:
# ============================================================
# QUADRANT SEGMENTATION
# Split movies into 4 buckets based on median split:
#
# Q1 — High Sentiment + High Longevity = SAFE BET (license long term)
# Q2 — Low Sentiment  + High Longevity = HIDDEN GEM (undervalued)
# Q3 — High Sentiment + Low Longevity  = OVERHYPED (short term only)
# Q4 — Low Sentiment  + Low Longevity  = AVOID
# ============================================================

sentiment_median = df["sentiment_compound"].median()
longevity_median = df["longevity_score"].median()

print(f"Sentiment median: {sentiment_median:.4f}")
print(f"Longevity median: {longevity_median:.2f}")

def assign_quadrant(row):
    high_sentiment = row["sentiment_compound"] >= sentiment_median
    high_longevity = row["longevity_score"]    >= longevity_median

    if high_sentiment and high_longevity:
        return "Safe Bet"
    elif not high_sentiment and high_longevity:
        return "Hidden Gem"
    elif high_sentiment and not high_longevity:
        return "Overhyped"
    else:
        return "Avoid"

df["quadrant"] = df.apply(assign_quadrant, axis=1)

print("\nQuadrant Distribution:")
print(df["quadrant"].value_counts())

Sentiment median: -0.1637
Longevity median: 4.89

Quadrant Distribution:
quadrant
Safe Bet      63
Avoid         63
Hidden Gem    62
Overhyped     62
Name: count, dtype: int64


In [6]:
# ============================================================
# CORE BUSINESS QUESTION:
# Does sentiment predict longevity independent of rating?
# ============================================================

from scipy import stats

# Sentiment vs Longevity
corr_sent_long, p_sent_long = stats.pearsonr(
    df["sentiment_compound"],
    df["longevity_score"]
)

# Rating vs Longevity
corr_rate_long, p_rate_long = stats.pearsonr(
    df["rating"],
    df["longevity_score"]
)

# Sentiment vs Rating
corr_sent_rate, p_sent_rate = stats.pearsonr(
    df["sentiment_compound"],
    df["rating"]
)

print("=" * 50)
print("CORRELATION RESULTS")
print("=" * 50)
print(f"\nSentiment vs Longevity:  r = {corr_sent_long:.4f}  (p = {p_sent_long:.4f})")
print(f"Rating vs Longevity:     r = {corr_rate_long:.4f}  (p = {p_rate_long:.4f})")
print(f"Sentiment vs Rating:     r = {corr_sent_rate:.4f}  (p = {p_sent_rate:.4f})")

print("\nInterpretation:")
print(f"  Sentiment predicts longevity: {'YES' if p_sent_long < 0.05 else 'WEAK — use cautiously'}")
print(f"  Rating predicts longevity:    {'YES' if p_rate_long < 0.05 else 'WEAK — use cautiously'}")

CORRELATION RESULTS

Sentiment vs Longevity:  r = -0.0112  (p = 0.8603)
Rating vs Longevity:     r = 0.0845  (p = 0.1828)
Sentiment vs Rating:     r = -0.1299  (p = 0.0401)

Interpretation:
  Sentiment predicts longevity: WEAK — use cautiously
  Rating predicts longevity:    WEAK — use cautiously


In [7]:
# ============================================================
# OUTLIERS — The most interesting business insights
# These are the movies that defy expectations
# ============================================================

# Overhyped — High sentiment but low longevity
# These are BAD licensing bets despite positive buzz
overhyped = df[df["quadrant"] == "Overhyped"].sort_values(
    "sentiment_compound", ascending=False
).head(10)[["title", "year", "rating", "sentiment_compound", "longevity_score", "genre"]]

# Hidden Gems — Low sentiment but high longevity
# These are UNDERVALUED licensing opportunities
hidden_gems = df[df["quadrant"] == "Hidden Gem"].sort_values(
    "longevity_score", ascending=False
).head(10)[["title", "year", "rating", "sentiment_compound", "longevity_score", "genre"]]

print("TOP 10 OVERHYPED MOVIES (High Sentiment, Low Longevity)")
print("=" * 60)
print(overhyped.to_string(index=False))

print("\n\nTOP 10 HIDDEN GEMS (Low Sentiment, High Longevity)")
print("=" * 60)
print(hidden_gems.to_string(index=False))

TOP 10 OVERHYPED MOVIES (High Sentiment, Low Longevity)
                title  year  rating  sentiment_compound  longevity_score                      genre
  Hachi: A Dog's Tale  2009     8.1              0.9859             2.56   Biography, Drama, Family
         Hotel Rwanda  2004     8.1              0.9850             2.19  Biography, Drama, History
             The Help  2011     8.1              0.9681             4.72                      Drama
        Before Sunset  2004     8.1              0.9524             1.71             Drama, Romance
       Before Sunrise  1995     8.1              0.9492             1.39     Comedy, Drama, Romance
 Beauty and the Beast  1991     8.0              0.9477             1.69 Animation, Family, Fantasy
It's a Wonderful Life  1946     8.6              0.9473             0.65     Drama, Family, Fantasy
               Dangal  2016     8.3              0.9360             3.31   Action, Biography, Drama
         Donnie Darko  2001     8.0         

In [8]:
# ============================================================
# GENRE ANALYSIS
# Does sentiment predict longevity better in some genres?
# ============================================================

# Extract primary genre only
df["primary_genre"] = df["genre"].str.split(",").str[0].str.strip()

genre_analysis = df.groupby("primary_genre").agg(
    movie_count        = ("title", "count"),
    avg_sentiment      = ("sentiment_compound", "mean"),
    avg_longevity      = ("longevity_score", "mean"),
    avg_rating         = ("rating", "mean"),
).round(3)

# Only keep genres with 3+ movies for statistical reliability
genre_analysis = genre_analysis[genre_analysis["movie_count"] >= 3]
genre_analysis = genre_analysis.sort_values("avg_longevity", ascending=False)

print("GENRE BREAKDOWN — Sentiment vs Longevity")
print("=" * 60)
print(genre_analysis.to_string())

GENRE BREAKDOWN — Sentiment vs Longevity
               movie_count  avg_sentiment  avg_longevity  avg_rating
primary_genre                                                       
Action                  46         -0.162         13.380       8.159
Biography               24          0.193          8.724       8.188
Adventure               51          0.168          7.150       8.273
Crime                   31         -0.588          6.540       8.439
Drama                   65         -0.042          5.945       8.275
Comedy                  24          0.215          4.188       8.225
Mystery                  3         -0.873          3.553       8.200
Animation                3          0.224          2.770       8.300
Horror                   3         -0.709          1.710       8.267


In [9]:
# ============================================================
# BUSINESS RECOMMENDATIONS
# Written as if presenting to a Netflix licensing team
# ============================================================

# Best genre for long term licensing
best_genre = genre_analysis["avg_longevity"].idxmax()
worst_genre = genre_analysis["avg_longevity"].idxmin()

# Safe bet count
safe_bets   = len(df[df["quadrant"] == "Safe Bet"])
hidden_gem_count = len(df[df["quadrant"] == "Hidden Gem"])
overhyped_count  = len(df[df["quadrant"] == "Overhyped"])

print("=" * 60)
print("BUSINESS RECOMMENDATIONS FOR STREAMING LICENSING")
print("=" * 60)

print(f"""
RECOMMENDATION 1 — Prioritize {best_genre} for Long Term Licensing
{best_genre} films show the highest longevity scores in our dataset,
meaning audiences continue engaging with them decades after release.
These represent the strongest long term licensing ROI.

RECOMMENDATION 2 — Do Not License Based on Sentiment Alone
{overhyped_count} movies in our dataset show high critic sentiment
but below-median longevity. Positive buzz at release does not
guarantee sustained audience engagement. Sentiment should be one
signal among many, not the primary licensing criterion.

RECOMMENDATION 3 — Investigate Hidden Gems for Undervalued Deals
{hidden_gem_count} movies show low sentiment scores but high longevity,
meaning they outperformed critic expectations significantly.
These titles are likely underpriced in licensing negotiations
and represent high ROI opportunities for catalogue acquisition.
""")

BUSINESS RECOMMENDATIONS FOR STREAMING LICENSING

RECOMMENDATION 1 — Prioritize Action for Long Term Licensing
Action films show the highest longevity scores in our dataset,
meaning audiences continue engaging with them decades after release.
These represent the strongest long term licensing ROI.

RECOMMENDATION 2 — Do Not License Based on Sentiment Alone
62 movies in our dataset show high critic sentiment
but below-median longevity. Positive buzz at release does not
guarantee sustained audience engagement. Sentiment should be one
signal among many, not the primary licensing criterion.

RECOMMENDATION 3 — Investigate Hidden Gems for Undervalued Deals
62 movies show low sentiment scores but high longevity,
meaning they outperformed critic expectations significantly.
These titles are likely underpriced in licensing negotiations
and represent high ROI opportunities for catalogue acquisition.



In [10]:
# Save the fully enriched dataset with all new columns
output_path = BASE_DIR / "data" / "processed" / "imdb_final.csv"
df.to_csv(output_path, index=False)

print(f"Saved enriched dataset to: {output_path}")
print(f"Total columns: {len(df.columns)}")
print(f"Columns: {list(df.columns)}")

Saved enriched dataset to: C:\Users\pranav sandanamyna\OneDrive\Desktop\criticlens\data\processed\imdb_final.csv
Total columns: 29
Columns: ['rank', 'title', 'year', 'rating', 'votes', 'genre', 'runtime_minutes', 'imdb_id', 'imdb_url', 'weighted_score', 'plot', 'awards', 'metascore', 'imdb_rating', 'box_office', 'director', 'language', 'country', 'sentiment_compound', 'sentiment_positive', 'sentiment_negative', 'sentiment_neutral', 'sentiment_intensity', 'sentiment_label', 'age_years', 'votes_per_year', 'longevity_score', 'quadrant', 'primary_genre']
